# Día 3 · Variables y regla anti-fuga

**Objetivo de hoy:** revisar las 151 columnas del archivo original y decidir, una por una, si el modelo puede usarla. El resultado es `config/variables.csv`, el "contrato" de variables del modelo.

**Por qué importa:** la **fuga de información** (*data leakage*) es usar en el modelo datos que no existían en el momento de la decisión. Un modelo con fuga da un AUC espectacular en el notebook y se cae en producción. Es el error más común en modelos de riesgo, y fue uno de los ítems que fallaste en la prueba de calibración del diagnóstico de tu ruta. Hoy lo vas a **ver** con números.

**Herramienta nueva: DuckDB.** El Día 1 cargamos solo 28 columnas con pandas para cuidar la memoria. Hoy necesitamos las 151. En vez de pandas, usaremos DuckDB: un motor SQL que corre dentro de Python, sin servidor, y que lee CSV y Parquet directamente. Ventajas:
- Escribes **SQL**, que es tu lenguaje fuerte.
- Usa todos los núcleos del procesador y procesa por partes, sin cargar todo en RAM.
- Puede consultar un Parquet como si fuera una tabla, sin cargarlo a pandas.

**Sobre la GPU:** hoy no aplica. El trabajo pesado es leer y descomprimir el CSV, y eso depende del disco y la CPU. DuckDB ya aprovecha todos los núcleos de tu procesador.

**Plan de la hora**
1. Conectar DuckDB y crear un Parquet con todas las columnas (celdas 1–2, ~3 min de proceso).
2. Esquema y nulos por columna y por año (celdas 3–4).
3. Clasificar cada columna (celdas 5–6).
4. Ver la fuga con números: AUC de cada variable sola (celdas 7–8).
5. Guardar `variables.csv` (celda 9).

## Celda 1 · Conectar DuckDB

**Qué hace:** importa librerías, define rutas y abre una conexión DuckDB en memoria.

**Por qué así:**
- `duckdb.connect()` sin argumentos crea una base **en memoria**: no deja archivos `.duckdb` en el disco. No los necesitamos porque nuestros datos viven en Parquet. Si quisieras una base persistente: `duckdb.connect("archivo.duckdb")`.
- **`.as_posix()`** convierte la ruta de Windows (`..\\data\\raw`) a barras normales (`../data/raw`). Las rutas van dentro de un texto SQL, y las barras invertidas `\\` dentro de un texto pueden interpretarse como caracteres especiales.
- **`con.execute(sql)`** ejecuta una consulta; después pides el resultado con `.df()` (como DataFrame de pandas) o `.fetchone()` (una fila). Existe también `con.sql(...)`, pero cambió de comportamiento entre versiones de DuckDB; `execute` funciona igual en todas.
- **`config/`** es una carpeta nueva, fuera de `data/`. `variables.csv` no son datos sino una **decisión de diseño** del modelo, así que sí debe ir a Git (y `data/` está en el `.gitignore`).

In [1]:
from pathlib import Path
import time
import duckdb
import numpy as np
import pandas as pd

RAW      = Path("../data/raw/accepted_2007_to_2018Q4.csv.gz").as_posix()
FULL_PQ  = Path("../data/interim/loans_36m_full.parquet").as_posix()
CONFIG   = Path("../config"); CONFIG.mkdir(exist_ok=True)

con = duckdb.connect()
print("DuckDB", duckdb.__version__, "| hilos:", con.execute("SELECT current_setting('threads')").fetchone()[0])

DuckDB 2.0.0 | hilos: 24


## Celda 2 · Crear un Parquet con todas las columnas (SQL)

**Qué hace:** lee el CSV comprimido, filtra préstamos a 36 meses de 2012–2015 (el mismo filtro del Día 1) y escribe el resultado a Parquet. Todo en una sola instrucción SQL: `COPY (consulta) TO archivo`.

**Por qué así, parte por parte:**
- **`read_csv(...)`** lee el `.gz` directamente, igual que pandas.
- **`sample_size = -1`**: DuckDB adivina el tipo de cada columna mirando una muestra de filas. Con `-1` mira **todas**. Es más lento, pero evita un error clásico: que una columna vacía en las primeras 20.000 filas se declare como texto y después aparezcan números. Con 151 columnas, muchas de ellas vacías en los primeros años, vale la pena.
- **`null_padding = true`**: el archivo termina con un par de filas "basura" (totales de Lending Club) que tienen menos columnas. En vez de fallar, DuckDB las completa con vacíos, y el filtro de fecha las elimina.
- **`parallel = false`**: la columna `desc` tiene textos con saltos de línea dentro de comillas. El lector paralelo de DuckDB no puede combinar eso con `null_padding` (lo comprobé probando el notebook: da error). Como un `.gz` se descomprime en un solo hilo de todos modos, no perdemos velocidad.
- **`try_strptime(issue_d, '%b-%Y')`** convierte "Dec-2015" en fecha. El prefijo `try_` devuelve `NULL` en vez de error si el texto no es una fecha (las filas basura). Es el equivalente de `pd.to_datetime(..., errors="coerce")`.
- **`trim(term)`**: igual que `.str.strip()` el Día 1, por el espacio inicial en " 36 months".
- **`COMPRESSION zstd`**: compresión moderna. Archivo más pequeño que con la opción por defecto y lectura igual de rápida.
- **El `if not exists`** evita repetir los ~3 minutos si vuelves a correr el notebook.

Es el mismo patrón ELT que usas en IWG: extraer de la fuente cruda, cargar a un formato analítico y transformar con SQL.

In [2]:
if not Path(FULL_PQ).exists():
    t0 = time.perf_counter()
    con.execute(f"""
        COPY (
            SELECT
                *,
                try_strptime(CAST(issue_d AS VARCHAR), '%b-%Y')::DATE AS issue_dt
            FROM read_csv('{RAW}', sample_size = -1, null_padding = true, parallel = false)
            WHERE trim(term) = '36 months'
              AND year(try_strptime(CAST(issue_d AS VARCHAR), '%b-%Y')) BETWEEN 2012 AND 2015
        ) TO '{FULL_PQ}' (FORMAT parquet, COMPRESSION zstd)
    """)
    print(f"Parquet creado en {time.perf_counter() - t0:.0f} s")

con.execute(f"SELECT count(*) AS filas FROM '{FULL_PQ}'").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet creado en 27 s


,filas
0,589635


**Control:** deben salir **589.635** filas, las mismas del Día 1. Si coincide, los dos caminos (pandas y DuckDB) llegaron al mismo resultado: es una validación cruzada de tu propio proceso de carga.

Fíjate en la última línea: `FROM '{FULL_PQ}'`. DuckDB consulta el Parquet como si fuera una tabla, sin cargarlo a pandas.

## Celda 3 · Esquema y perfil de cada columna

**Qué hace:** `SUMMARIZE` es una instrucción de DuckDB que devuelve, para cada columna: tipo, mínimo, máximo, valores únicos aproximados, promedio y **porcentaje de nulos**. Es un perfilado completo en una línea.

**Por qué así:** en pandas tendrías que combinar `df.dtypes`, `df.isna().mean()`, `df.nunique()` y `df.describe()`, y además cargar las 151 columnas en RAM. Aquí DuckDB lo calcula leyendo el Parquet por partes.

`.df()` convierte el resultado de DuckDB en un DataFrame de pandas. Es pequeño (una fila por columna), así que no hay problema de memoria.

In [3]:
perfil = con.execute(f"SUMMARIZE SELECT * FROM '{FULL_PQ}'").df()
perfil = perfil[["column_name", "column_type", "approx_unique", "null_percentage", "min", "max"]]
perfil["null_percentage"] = perfil["null_percentage"].astype(float)
print(len(perfil), "columnas")
perfil.sort_values("null_percentage", ascending=False).head(25)

152 columnas


,column_name,column_type,approx_unique,null_percentage,min,max
1,member_id,VARCHAR,0,100.00,NaN,NaN
126,sec_app_collections_12_mths_ex_med,DOUBLE,0,100.00,NaN,NaN
125,sec_app_chargeoff_within_12_mths,DOUBLE,0,100.00,NaN,NaN
124,sec_app_num_rev_accts,DOUBLE,0,100.00,NaN,NaN
123,sec_app_open_act_il,DOUBLE,0,100.00,NaN,NaN
122,sec_app_revol_util,DOUBLE,0,100.00,NaN,NaN
121,sec_app_open_acc,DOUBLE,0,100.00,NaN,NaN
120,sec_app_mort_acc,DOUBLE,0,100.00,NaN,NaN
127,sec_app_mths_since_last_major_derog,DOUBLE,0,100.00,NaN,NaN
118,sec_app_earliest_cr_line,VARCHAR,0,100.00,NaN,NaN


## Celda 4 · Nulos por año de originación

**Qué hace:** calcula el % de nulos de **cada columna en cada año**. Construye la consulta SQL desde Python, porque escribir 151 columnas a mano no tiene sentido.

**Por qué importa:** Lending Club agregó variables con el tiempo. Una columna puede estar 100 % vacía en 2012–2013 (desarrollo) y llena en 2015 (producción). El modelo **no puede** usarla, porque no la vio al entrenar. El % de nulos global (celda 3) esconde esto; por año se ve.

**Detalles del código:**
- **Generar SQL con un *f-string***: `", ".join(...)` arma una lista de expresiones, una por columna. Es una técnica muy común; la usarás en pipelines donde las columnas cambian.
- **Comillas dobles alrededor del nombre** (`"{c}"`): en SQL, las comillas dobles marcan un nombre de columna y las simples un texto. Así funciona aunque una columna se llame como una palabra reservada (por ejemplo, `desc`, que sí existe en este archivo).
- **`("col" IS NULL)::INT`** da 1 si es nulo y 0 si no; su promedio (`avg`) es la proporción de nulos. Es el mismo truco del `mean()` sobre 0/1 del Día 2.
- **`.T`** transpone: columnas del archivo como filas y años como columnas, que se lee mejor.

In [4]:
columnas = [c for c in perfil["column_name"] if c != "issue_dt"]
expr = ", ".join(f'avg(("{c}" IS NULL)::INT) AS "{c}"' for c in columnas)

nulos_anio = con.execute(f"""
    SELECT year(issue_dt) AS anio, {expr}
    FROM '{FULL_PQ}'
    GROUP BY anio ORDER BY anio
""").df().set_index("anio").T

nulos_anio["nulos_desarrollo"] = (
    con.execute(f"SELECT {expr} FROM '{FULL_PQ}' WHERE year(issue_dt) <= 2013").df().T.iloc[:, 0]
)

# columnas cuyo % de nulos cambia mucho entre años
cambio = nulos_anio[[2012, 2013, 2014, 2015]].max(axis=1) - nulos_anio[[2012, 2013, 2014, 2015]].min(axis=1)
nulos_anio[cambio > 0.2].style.format("{:.0%}").background_gradient(cmap="Oranges", axis=None)

anio,2012,2013,2014,2015,nulos_desarrollo
desc,38%,63%,93%,100%,56%
mths_since_last_major_derog,90%,80%,71%,70%,83%
tot_coll_amt,52%,0%,0%,0%,16%
tot_cur_bal,52%,0%,0%,0%,16%
total_rev_hi_lim,52%,0%,0%,0%,16%
avg_cur_bal,52%,0%,0%,0%,16%
mo_sin_old_il_acct,54%,4%,4%,4%,19%
mo_sin_old_rev_tl_op,52%,0%,0%,0%,16%
mo_sin_rcnt_rev_tl_op,52%,0%,0%,0%,16%
mo_sin_rcnt_tl,52%,0%,0%,0%,16%


## Celda 5 · Reglas de clasificación

**Qué hace:** define las listas y reglas que deciden qué columnas puede usar el modelo, con la razón de cada decisión.

**Por qué con listas escritas a mano y no automáticas:** la fuga es un concepto **de negocio**: hay que saber *cuándo* se genera cada dato. Ninguna estadística te dice que `recoveries` (lo recuperado después del castigo) no existía al aprobar el crédito. Por eso las reglas de fuga van explícitas, con su razón, y en la celda 7 las comprobamos con números.

**Las categorías, en orden de prioridad:**

| Categoría | Ejemplos | Decisión |
|---|---|---|
| objetivo | `loan_status` | excluir (es lo que predecimos) |
| identificador / texto libre | `id`, `url`, `emp_title`, `desc` | excluir: no generalizan o requieren NLP |
| fuga | pagos, recobros, último FICO, planes de alivio | **excluir siempre** |
| diseño de la muestra | `term`, `issue_d` | excluir: ya los usamos para filtrar y partir |
| score de Lending Club | `grade`, `sub_grade`, `int_rate`, `installment` | **benchmark**, no variable |
| no disponible en desarrollo | ≥ 99 % nulos en 2012–2013 | excluir |
| constante | un solo valor | excluir: no aporta información |
| permitida | ingresos, DTI, FICO al aprobar, buró | usar |

**Decisión sobre el score de Lending Club:** `grade` e `int_rate` son el resultado del modelo de riesgo de Lending Club. Usarlos es válido (se conocen al aprobar), pero tu modelo terminaría copiando el de ellos. Vamos a hacer lo que haría un banco que evalúa su propio riesgo: construir el modelo **sin** ellos y usar `grade` como **benchmark**. Si nuestro modelo le gana a `grade`, aporta valor. Es una comparación muy buena para contar en una entrevista.

**Nulos parciales no son motivo de exclusión:** `mths_since_last_delinq` (meses desde la última mora) está vacío en ~50 % de los casos porque esas personas **nunca** han estado en mora. Ese vacío es información. El Día 6, con WoE, el vacío tendrá su propio tramo.

In [5]:
OBJETIVO = {"loan_status"}
IDENTIFICADOR_TEXTO = {"id", "member_id", "url", "desc", "emp_title", "title", "zip_code"}
DISENO = {"term", "issue_d", "issue_dt"}
SCORE_LC = {"grade", "sub_grade", "int_rate", "installment"}

FUGA = {
    # pagos y saldos después del desembolso
    "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "pymnt_plan",
    # información de buró consultada después
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low",
    # acuerdos de pago y planes de alivio (solo existen si el crédito se deterioró)
    "debt_settlement_flag", "debt_settlement_flag_date", "deferral_term",
    "payment_plan_start_date", "orig_projected_additional_accrued_interest",
    # monto financiado por inversionistas: se conoce después de publicar el préstamo
    "funded_amnt_inv",
}
PREFIJOS_FUGA = ("hardship_", "settlement_")

def clasificar(col, nulos_dev, unicos):
    if col in OBJETIVO:              return "objetivo", "Es la variable que predecimos"
    if col in IDENTIFICADOR_TEXTO:   return "identificador/texto", "Identificador o texto libre"
    if col in FUGA or col.startswith(PREFIJOS_FUGA):
                                     return "fuga", "Se genera después de aprobar el crédito"
    if col in DISENO:                return "diseño", "Usada para filtrar y partir la muestra"
    if col in SCORE_LC:              return "score_LC", "Resultado del modelo de Lending Club: benchmark"
    if nulos_dev >= 0.99:            return "no_disponible", f"{nulos_dev:.0%} nulos en desarrollo"
    if unicos <= 1:                  return "constante", "Un solo valor"
    return "permitida", "Conocida al momento de aprobar"

DECISION = {"permitida": "usar", "score_LC": "benchmark"}   # todo lo demás: excluir

## Celda 6 · Aplicar las reglas

**Qué hace:** cruza el perfil (celda 3) con los nulos de desarrollo (celda 4) y aplica `clasificar` a cada columna.

**Por qué así:**
- **`.apply(..., axis=1)`** ejecuta la función fila por fila. Con 151 filas es instantáneo. En tablas de millones de filas evitarías `apply` (es lento) y usarías operaciones vectorizadas, pero aquí la claridad importa más.
- **`result_type="expand"`** convierte la tupla `(categoría, razón)` que devuelve la función en dos columnas.
- **`.map(DECISION).fillna("excluir")`**: un diccionario corto con las excepciones y un valor por defecto. Mismo patrón seguro del Día 2.

In [6]:
vars_df = perfil.rename(columns={"column_name": "columna", "column_type": "tipo",
                                 "null_percentage": "nulos_pct_total"}).copy()
vars_df["nulos_pct_total"] = vars_df["nulos_pct_total"] / 100
vars_df["nulos_desarrollo"] = vars_df["columna"].map(nulos_anio["nulos_desarrollo"]).fillna(0)

vars_df[["categoria", "razon"]] = vars_df.apply(
    lambda r: clasificar(r["columna"], r["nulos_desarrollo"], r["approx_unique"]),
    axis=1, result_type="expand",
)
vars_df["decision"] = vars_df["categoria"].map(DECISION).fillna("excluir")

vars_df.groupby(["decision", "categoria"]).size().rename("columnas").to_frame()

columnas
decision  categoria                    
benchmark score_LC                    4
excluir   constante                   2
          diseño                      3
          fuga                       39
          identificador/texto         7
          no_disponible              30
          objetivo                    1
usar      permitida                  66

## Celda 7 · Ver la fuga con números: AUC de cada variable sola

**Qué hace:** para cada variable numérica, mide qué tan bien separa buenos de malos **ella sola**, con el AUC. Usa solo la muestra de desarrollo (2012–2013) y la población del Día 2 (sin indeterminados).

**Por qué funciona como detector de fuga:** en riesgo de crédito, una variable conocida al aprobar casi nunca supera un AUC de 0,70 sola (el FICO anda por 0,60–0,65). Si una variable da 0,90 o más, casi seguro "conoce el futuro". Es la prueba estadística que confirma las reglas de negocio de la celda 5.

**Detalles del código:**
- **Traemos solo las columnas numéricas** a pandas, filtradas en SQL. DuckDB hace el filtro y pandas recibe solo lo necesario (~144.000 filas).
- **`CASE WHEN ... THEN 1`** crea `malo` en SQL con la misma regla del Día 2.
- **Nulos**: el AUC no acepta vacíos, así que los llenamos con la mediana. Es una simplificación válida solo para este diagnóstico; en el modelo los trataremos mejor.
- **`max(auc, 1 - auc)`**: una variable puede separar "al revés" (más valor = menos riesgo, como el FICO). Las dos direcciones son útiles, así que tomamos la más alta. 0,5 = no separa nada.
- **Gini = 2 × AUC − 1**, la misma medida expresada como se usa en banca.

In [7]:
from sklearn.metrics import roc_auc_score

numericas = vars_df.loc[vars_df["tipo"].str.contains("INT|DOUBLE|FLOAT|DECIMAL|BIGINT"), "columna"].tolist()
numericas = [c for c in numericas if c not in {"id", "member_id"}]
sel = ", ".join(f'"{c}"' for c in numericas)

dev = con.execute(f"""
    SELECT {sel},
           CASE WHEN loan_status IN ('Charged Off', 'Default') THEN 1 ELSE 0 END AS malo
    FROM '{FULL_PQ}'
    WHERE year(issue_dt) <= 2013
      AND loan_status IN ('Fully Paid', 'Charged Off', 'Default')
""").df()
print(dev.shape, f"tasa de malos: {dev['malo'].mean():.2%}")

filas = []
for c in numericas:
    x = dev[c]
    if x.notna().sum() == 0 or x.nunique() < 2:
        continue
    auc = roc_auc_score(dev["malo"], x.fillna(x.median()))
    auc = max(auc, 1 - auc)
    filas.append({"columna": c, "auc": auc, "gini": 2 * auc - 1})

auc_df = (pd.DataFrame(filas)
          .merge(vars_df[["columna", "categoria"]], on="columna")
          .sort_values("auc", ascending=False))
auc_df.head(20).style.format({"auc": "{:.3f}", "gini": "{:.3f}"}).background_gradient(subset="auc", cmap="Reds")

(143892, 113) tasa de malos: 12.70%


,columna,auc,gini,categoria
23,recoveries,0.971,0.942,fuga
27,last_fico_range_low,0.886,0.772,fuga
26,last_fico_range_high,0.886,0.772,fuga
24,collection_recovery_fee,0.861,0.722,fuga
20,total_rec_prncp,0.835,0.669,fuga
25,last_pymnt_amnt,0.767,0.534,fuga
18,total_pymnt,0.766,0.532,fuga
19,total_pymnt_inv,0.766,0.531,fuga
3,int_rate,0.638,0.276,score_LC
9,fico_range_high,0.596,0.192,permitida


## Celda 8 · Comparar por categoría

**Qué hace:** resume el AUC máximo y mediano por categoría.

**Qué deberías ver:** las variables de **fuga** arriba, con AUC muy alto (los pagos recibidos y el último FICO separan casi perfecto porque reflejan lo que ya pasó). Las **permitidas**, con AUC moderados. El **score de LC** (`int_rate`), en medio: es un buen modelo, pero no adivina el futuro.

Si una variable que marcamos como permitida aparece con AUC > 0,80, hay que revisarla: puede ser una fuga que se nos escapó. Ese es el valor de combinar las reglas de negocio con la prueba estadística.

In [8]:
auc_df.groupby("categoria")["auc"].agg(["count", "max", "median"]).sort_values("max", ascending=False).style.format({"max": "{:.3f}", "median": "{:.3f}"})

,count,max,median
categoria,,,
fuga,14,0.971,0.766
score_LC,2,0.638,0.573
permitida,58,0.596,0.524


## Celda 9 · Guardar el contrato de variables

**Qué hace:** guarda `config/variables.csv` con columna, tipo, nulos, categoría, razón, decisión y AUC individual.

**Por qué es importante:** este archivo es la **documentación de las decisiones** del modelo. En un banco, el área de validación de modelos pediría exactamente esta tabla: qué variables se evaluaron, cuáles se excluyeron y por qué. Desde los próximos días, el código leerá de aquí la lista de variables a usar, en vez de escribirlas a mano en cada notebook. Si cambias una decisión, la cambias en un solo lugar.

Después de guardar, haz commit: este archivo **sí** va a Git.

In [9]:
salida = vars_df.merge(auc_df[["columna", "auc"]], on="columna", how="left")
salida = salida[["columna", "tipo", "nulos_pct_total", "nulos_desarrollo", "categoria", "razon", "decision", "auc"]]
salida = salida.sort_values(["decision", "categoria", "columna"])
salida.to_csv(CONFIG / "variables.csv", index=False, float_format="%.4f")

print("Variables para el modelo:", (salida["decision"] == "usar").sum())
print(salida.loc[salida["decision"] == "usar", "columna"].tolist())

Variables para el modelo: 66
['acc_now_delinq', 'acc_open_past_24mths', 'addr_state', 'annual_inc', 'application_type', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'delinq_2yrs', 'delinq_amnt', 'dti', 'earliest_cr_line', 'emp_length', 'fico_range_high', 'fico_range_low', 'funded_amnt', 'home_ownership', 'initial_list_status', 'inq_last_6mths', 'loan_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_last_delinq', 'mths_since_last_major_derog', 'mths_since_last_record', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'open_acc', 'pct_tl_nvr_dlq', 'perce

---
## Para la bitácora

1. ¿Cuántas columnas quedaron en cada categoría? ¿Cuántas para usar?
2. Las 5 variables con mayor AUC y su categoría. ¿Se confirmó la fuga?
3. ¿Qué AUC tienen `int_rate` y `fico_range_low`? (Son la referencia para saber si nuestro modelo es bueno.)
4. ¿Alguna variable que aparece llena en 2015 y vacía en 2012–2013?
5. Una variable que te haya sorprendido.

```powershell
git add config/variables.csv notebooks/03_variables.ipynb bitacora.md
git commit -m "Día 3: clasificación de variables y control de fuga"
```